# PGC Low-Trap Scan Analysis

Interactive analysis of `PGCLowTrapScan` runs. Reuses the BLAS-free helpers from
`analyze_pgc_lowtrap.py`. Two 2-D heatmaps over (cooling-DP frequency, amplitude):

- **Panel A** — recapture at the actual release time = the **temperature proxy** (higher =
  colder). Its **peak = the best low-trap PGC (f, amp)** (marked with a red star).
- **Panel B** — retention at release=0 = "did the PGC keep the atom" (a sanity map).

Run with the **working-BLAS `python`** kernel (numpy 2.x / h5py / mpl 3.x); matplotlib
crashes in the artiq7_claude env.

In [ ]:
import sys, os, glob
from pathlib import Path
_ANALYSIS_DIR = r"C:\Networking Experiment\Artiq7_TEST_NEW\useful_codes\repository\analysis"
if _ANALYSIS_DIR not in sys.path:
    sys.path.insert(0, _ANALYSIS_DIR)
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%load_ext autoreload
%autoreload 2
from analyze_pgc_lowtrap import (
    RESULTS_DIR, find_run, load_run, recapture_table, build_grids, wilson,
)
print("Results dir:", RESULTS_DIR)

## 1. List recent runs

In [ ]:
files = sorted(glob.glob(str(RESULTS_DIR / "**" / "*-PGCLowTrapScan.h5"), recursive=True),
               key=lambda f: int(os.path.basename(f).split("-")[0]))
if not files:
    print("No *-PGCLowTrapScan.h5 found yet.")
for f in files[-12:]:
    rid = int(os.path.basename(f).split("-")[0])
    d, s, _ = load_run(f)
    n = len(np.atleast_1d(d.get("pgc_f", [])))
    nl = int(np.sum(np.atleast_1d(d.get("pgc_atom_loaded", [0])).astype(int)))
    print(f"RID {rid:>5} | {n:>6} pts | {nl:>5} loaded | release={s.get('release_us','?')} us | "
          f"pgc_dur={s.get('pgc_duration_us','?')} us")

## 2. Select a run

Set `RID` to an int, or `None` for the latest.

In [ ]:
RID = None   # <-- int or None (latest)

path = find_run(RID)
data, snap, rid = load_run(path)
fs, amps, M_rec, M_ret, rel_probe, best_fa, best_p = build_grids(data)
fs_mhz = [f / 1e6 for f in fs]

print("File :", os.path.basename(path))
print(f"RID {rid} | {len(fs)} freq x {len(amps)} amp | release probe = {rel_probe:g} us")
print("freqs (MHz):", [round(x, 4) for x in fs_mhz])
print("amps       :", amps)

## 3. Panel A — recapture @ release (temperature proxy) → the best (f, amp)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
im = ax.pcolormesh(fs_mhz, amps, M_rec, shading="nearest", cmap="viridis", vmin=0.0, vmax=1.0)
fig.colorbar(im, ax=ax, label="recapture")
if best_fa is not None:
    ax.plot(best_fa[0] / 1e6, best_fa[1], "*", color="red", ms=18,
            markeredgecolor="white", label="best")
    ax.legend(loc="upper right")
ax.set_xlabel("cooling-DP frequency (MHz)")
ax.set_ylabel("cooling-DP amplitude")
ax.set_title(f"Recapture @ {rel_probe:g} us (colder = higher) — RID {rid}")
plt.show()

if best_fa is not None:
    print(f"BEST low-trap PGC: f = {best_fa[0]/1e6:.4f} MHz, amp = {best_fa[1]:.3f}  "
          f"(recapture {best_p:.3f} @ {rel_probe:g} us)")
    print(f"baseline deep-PGC: f = {float(snap.get('f_cooling_deep_PGC', 0))/1e6:.4f} MHz, "
          f"amp = {snap.get('amp_cooling_deep_PGC', '?')}")
else:
    print("No recapture data at the probe release -- check release_us / loading.")

## 4. Panel B — retention @ 0 us (did the PGC keep the atom)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
im = ax.pcolormesh(fs_mhz, amps, M_ret, shading="nearest", cmap="viridis", vmin=0.0, vmax=1.0)
fig.colorbar(im, ax=ax, label="retention @ 0 us")
ax.set_xlabel("cooling-DP frequency (MHz)")
ax.set_ylabel("cooling-DP amplitude")
ax.set_title("Retention @ 0 us (low region = atom lost during PGC)")
plt.show()

## 5. (optional) one-shot PNG via the script

`analyze(path)` writes the 2-panel PNG next to the `.h5`.

In [ ]:
# from analyze_pgc_lowtrap import analyze
# analyze(path)